# 08 - LangGraph com MCP (Model Context Protocol)

Caso prático: um **agente de clima**. A ferramenta de temperatura não está no notebook. Ela vive em um **servidor MCP** separado (`servidor_mcp_clima.py`), e o LangGraph a consome por esse protocolo.

Fluxo:

```
START → agente → (o LLM pediu ferramenta?) → ferramentas MCP → agente → ... → END
                       └── não: responde e termina
```

## Conceitos de MCP

- **MCP** é um protocolo aberto que padroniza como um LLM (ou um agente) descobre e usa ferramentas, recursos e prompts expostos por outro processo.
- **Servidor MCP**: processo que anuncia suas ferramentas (`tools/list`) e executa as chamadas (`tools/call`).
- **Cliente MCP**: quem conecta ao servidor. Aqui, o cliente roda dentro do notebook.
- **Transporte**: como cliente e servidor conversam. Este notebook usa HTTP (*Streamable HTTP*): o servidor fica no ar em uma URL e o cliente envia mensagens JSON-RPC para ela. O servidor também suporta `stdio`, em que o cliente inicia o servidor como subprocesso.

## Como o LangGraph entra

O LangGraph não fala MCP diretamente. O pacote `langchain-mcp-adapters` faz a ponte:

1. Conecta ao servidor MCP.
2. Lê as ferramentas anunciadas e as converte em ferramentas do LangChain (`BaseTool`).
3. Essas ferramentas entram no grafo como qualquer outra, com `bind_tools` e `ToolNode`.

Por isso, a estrutura do grafo é a mesma do notebook 04. Só muda de onde vêm as ferramentas.

## 1. Configuração

A chave e o modelo vêm do `.env` na raiz do projeto, como nos outros notebooks.

In [1]:
import asyncio
import json
import os
import sys
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

load_dotenv(find_dotenv())

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
)

## 2. O servidor MCP

O servidor está em `servidor_mcp_clima.py`, ao lado deste notebook. Ele usa `FastMCP`: cada função decorada com `@mcp.tool()` vira uma ferramenta anunciada ao cliente. Nome, descrição e tipos dos parâmetros viram o contrato que o LLM vê.

Abaixo, o código do servidor:

In [2]:
codigo_servidor = Path("servidor_mcp_clima.py").read_text(encoding="utf-8")
display(Markdown(f"```python\n{codigo_servidor}\n```"))

```python
"""Servidor MCP mínimo de exemplo: consulta de temperatura de cidades.

Usado pelo notebook 08-MCP - Langgraph.ipynb.

Modos de execução:
- http (padrão): o servidor fica no ar em http://127.0.0.1:8000/mcp.
  Rode em um terminal separado e deixe aberto enquanto usa o notebook.
- stdio: o cliente inicia o servidor como subprocesso e conversa pela entrada e saída padrão.

Os dados são simulados em um dicionário para o exemplo não depender de internet.

Atenção: no transporte stdio, o stdout é o canal do protocolo. Não use print() aqui.
"""

import argparse
import json

from mcp.server.fastmcp import FastMCP

TEMPERATURAS = {
    "são paulo": {"temperatura_c": 24.5, "condicao": "parcialmente nublado"},
    "rio de janeiro": {"temperatura_c": 29.0, "condicao": "ensolarado"},
    "curitiba": {"temperatura_c": 16.2, "condicao": "chuva fraca"},
    "porto alegre": {"temperatura_c": 19.8, "condicao": "nublado"},
}

mcp = FastMCP("clima", host="127.0.0.1", port=8000)


@mcp.tool()
def obter_temperatura(cidade: str) -> str:
    """Retorna a temperatura atual, em graus Celsius, e a condição do tempo de uma cidade brasileira."""
    dados = TEMPERATURAS.get(cidade.lower().strip())
    if dados is None:
        return json.dumps({"cidade": cidade, "erro": "cidade não encontrada"}, ensure_ascii=False)
    return json.dumps({"cidade": cidade, **dados}, ensure_ascii=False)


@mcp.tool()
def listar_cidades() -> str:
    """Lista as cidades disponíveis para consulta de temperatura."""
    return json.dumps(sorted(TEMPERATURAS), ensure_ascii=False)


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Servidor MCP de clima de exemplo.")
    parser.add_argument(
        "--transporte",
        choices=["http", "stdio"],
        default="http",
        help="http: servidor sempre no ar (padrão). stdio: iniciado pelo cliente.",
    )
    args = parser.parse_args()

    if args.transporte == "http":
        print("Servidor MCP em http://127.0.0.1:8000/mcp (Ctrl+C para encerrar)", flush=True)
        mcp.run(transport="streamable-http")
    else:
        mcp.run(transport="stdio")

```

## 3. Iniciando o servidor em outro terminal

O servidor roda como um processo separado, então precisa estar no ar **antes** de o notebook se conectar. Ele fica escutando em `http://127.0.0.1:8000/mcp` (transporte *Streamable HTTP*).

1. Abra um terminal (fora do Jupyter).
2. Entre na pasta deste notebook:

```bash
cd notebooks/02-Langgraph
```

3. Inicie o servidor com o ambiente do projeto:

```bash
uv run --project ../.. python servidor_mcp_clima.py
```

Você deve ver algo como:

```
Servidor MCP em http://127.0.0.1:8000/mcp (Ctrl+C para encerrar)
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
```

4. Deixe esse terminal aberto. Volte ao notebook e execute as células seguintes. Para parar o servidor, pressione `Ctrl+C` no terminal dele.

Se a porta 8000 já estiver em uso, feche o processo que a ocupa antes de iniciar o servidor.

A célula abaixo confere se o servidor está respondendo. Se mostrar "servidor não encontrado", volte ao passo 3.

In [3]:
import socket

try:
    with socket.create_connection(("127.0.0.1", 8000), timeout=2):
        print("Servidor MCP encontrado em http://127.0.0.1:8000/mcp")
except OSError:
    print("Servidor não encontrado. Inicie o servidor em outro terminal (veja o passo 3 acima).")

Servidor MCP encontrado em http://127.0.0.1:8000/mcp


## 4. Conectando ao servidor

`MultiServerMCPClient` recebe um dicionário com um ou mais servidores. Aqui há um só, chamado `clima`, com transporte `streamable_http` e a URL do servidor que você iniciou no terminal.

`get_tools()` faz o `tools/list` e devolve as ferramentas já convertidas para LangChain.

In [4]:
cliente_mcp = MultiServerMCPClient(
    {
        "clima": {
            "transport": "streamable_http",
            "url": "http://127.0.0.1:8000/mcp",
        }
    }
)

FERRAMENTAS = await cliente_mcp.get_tools()

for ferramenta in FERRAMENTAS:
    print(f"{ferramenta.name}")
    print(f"  descrição: {ferramenta.description}")
    print(f"  parâmetros: {ferramenta.args}")

obter_temperatura
  descrição: Retorna a temperatura atual, em graus Celsius, e a condição do tempo de uma cidade brasileira.
  parâmetros: {'cidade': {'title': 'Cidade', 'type': 'string'}}
listar_cidades
  descrição: Lista as cidades disponíveis para consulta de temperatura.
  parâmetros: {}


## 5. Chamada direta, sem LLM

Antes de envolver o LLM, vale ver o protocolo sozinho. `client.session("clima")` abre uma sessão MCP com o servidor, e `call_tool` faz o `tools/call`. É o mesmo caminho que o `ToolNode` vai usar mais adiante.

In [5]:
async with cliente_mcp.session("clima") as sessao:
    resultado = await sessao.call_tool("obter_temperatura", {"cidade": "Porto Alegre"})
    print(resultado.content[0].text)

    resultado = await sessao.call_tool("listar_cidades", {})
    print(resultado.content[0].text)

{"cidade": "Porto Alegre", "temperatura_c": 19.8, "condicao": "nublado"}
["curitiba", "porto alegre", "rio de janeiro", "são paulo"]


## 6. Montando o agente

A mesma estrutura do notebook 04:

- `llm_com_tools` é o LLM com as ferramentas MCP anexadas.
- O nó `agente` põe uma instrução de sistema na frente das mensagens, sem salvá-la no estado.
- O nó `ferramentas` é um `ToolNode`. Ele chama a ferramenta MCP correspondente à chamada pedida pelo LLM.
- `tools_condition` decide se o grafo vai para as ferramentas ou termina.

In [6]:
llm_com_tools = llm.bind_tools(FERRAMENTAS)

INSTRUCAO = (
    "Você é um assistente de clima. Use as ferramentas para consultar a temperatura. "
    "Não invente valores. Se uma cidade não for encontrada, diga isso. Responda em português."
)


async def agente(estado: MessagesState) -> dict:
    resposta = await llm_com_tools.ainvoke([SystemMessage(content=INSTRUCAO)] + estado["messages"])
    return {"messages": [resposta]}


grafo = StateGraph(MessagesState)
grafo.add_node("agente", agente)
grafo.add_node("ferramentas", ToolNode(FERRAMENTAS))

grafo.add_edge(START, "agente")
grafo.add_conditional_edges("agente", tools_condition, {"tools": "ferramentas", END: END})
grafo.add_edge("ferramentas", "agente")

app = grafo.compile()

## 7. Executando

A função `perguntar` roda o agente e mostra o caminho das mensagens, igual ao notebook 04:

- `→` é uma chamada de ferramenta MCP pedida pelo LLM.
- `←` é o resultado devolvido pelo servidor.
- `Resposta` é a mensagem final.

In [7]:
async def perguntar(texto: str) -> None:
    estado = await app.ainvoke({"messages": [HumanMessage(content=texto)]})

    linhas = [f"**Pergunta:** {texto}", ""]
    for mensagem in estado["messages"][1:]:
        if isinstance(mensagem, AIMessage) and mensagem.tool_calls:
            for chamada in mensagem.tool_calls:
                linhas.append(f"→ `{chamada['name']}({chamada['args']})`")
        elif isinstance(mensagem, ToolMessage):
            conteudo = mensagem.content
            if isinstance(conteudo, list):
                conteudo = " ".join(b["text"] for b in conteudo if isinstance(b, dict) and "text" in b)
            linhas.append(f"← `{conteudo}`")
        elif isinstance(mensagem, AIMessage):
            linhas.append("")
            linhas.append(f"**Resposta:** {mensagem.content}")
    display(Markdown("\n".join(linhas)))


await perguntar("Qual a temperatura em Curitiba e no Rio de Janeiro?")

**Pergunta:** Qual a temperatura em Curitiba e no Rio de Janeiro?

→ `obter_temperatura({'cidade': 'Curitiba'})`
→ `obter_temperatura({'cidade': 'Rio de Janeiro'})`
← `{"cidade": "Curitiba", "temperatura_c": 16.2, "condicao": "chuva fraca"}`
← `{"cidade": "Rio de Janeiro", "temperatura_c": 29.0, "condicao": "ensolarado"}`

**Resposta:** Aqui estão as temperaturas atuais:

- Curitiba: 16.2 °C, chuva fraca.
- Rio de Janeiro: 29.0 °C, ensolarado.

Quer mais alguma cidade ou previsão?

In [8]:
await perguntar("Está mais quente em São Paulo ou em Porto Alegre? Qual a diferença?")

**Pergunta:** Está mais quente em São Paulo ou em Porto Alegre? Qual a diferença?

→ `obter_temperatura({'cidade': 'São Paulo'})`
→ `obter_temperatura({'cidade': 'Porto Alegre'})`
← `{"cidade": "São Paulo", "temperatura_c": 24.5, "condicao": "parcialmente nublado"}`
← `{"cidade": "Porto Alegre", "temperatura_c": 19.8, "condicao": "nublado"}`

**Resposta:** Está mais quente em São Paulo. Agora:

- São Paulo: 24,5°C, parcialmente nublado  
- Porto Alegre: 19,8°C, nublado

Diferença: 4,7°C — São Paulo está 4,7°C mais quente que Porto Alegre.

Quer que eu traga a previsão para as próximas horas/dias ou a sensação térmica?

In [9]:
await perguntar("Quais cidades eu posso consultar? Me mostre a temperatura de Recife.")

**Pergunta:** Quais cidades eu posso consultar? Me mostre a temperatura de Recife.

→ `listar_cidades({})`
→ `obter_temperatura({'cidade': 'Recife'})`
← `["curitiba", "porto alegre", "rio de janeiro", "são paulo"]`
← `{"cidade": "Recife", "erro": "cidade não encontrada"}`

**Resposta:** Posso consultar as seguintes cidades: Curitiba, Porto Alegre, Rio de Janeiro e São Paulo.

Tentei obter a temperatura de Recife, mas o serviço retornou: cidade não encontrada. Não posso inventar valores — quer que eu consulte a temperatura de alguma das cidades disponíveis acima?

## 8. Conferindo o estado final

Como o estado é um `MessagesState`, a conversa inteira fica em `messages`. Cada chamada MCP e cada resultado aparecem ali.

In [10]:
estado = await app.ainvoke({"messages": [HumanMessage(content="Qual a temperatura em Curitiba?")]})

for mensagem in estado["messages"]:
    print(f"{mensagem.type:<6} | {mensagem.content!s:.80}")
    if isinstance(mensagem, AIMessage) and mensagem.tool_calls:
        print(f"       | tool_calls: {[c['name'] for c in mensagem.tool_calls]}")

human  | Qual a temperatura em Curitiba?
ai     | 
       | tool_calls: ['obter_temperatura']
tool   | [{'type': 'text', 'text': '{"cidade": "Curitiba", "temperatura_c": 16.2, "condic
ai     | A temperatura atual em Curitiba é de 16,2 °C, com chuva fraca. Deseja mais detal


## Observações

- A **descrição** de cada ferramenta vem do servidor, não do cliente. Quem mantém o servidor controla o que o LLM vê.
- O cliente não precisa conhecer a implementação. Trocar o corpo de `obter_temperatura` por uma chamada a uma API de clima real não muda nada do lado do grafo.
- O LLM não executa nada. Ele pede a chamada, e o `ToolNode` envia `tools/call` ao servidor.
- O servidor roda em outro processo e outro terminal, como um serviço. Por isso o notebook só precisa da URL. Para um servidor remoto, basta trocar a URL no dicionário de conexão.
- O transporte `stdio` também funciona: nesse caso o cliente inicia o servidor sozinho, e não é preciso abrir terminal. Para usá-lo, troque a conexão por `"transport": "stdio"`, `"command": sys.executable` e `"args": [caminho_do_servidor]`.
- O `ToolNode` recebe a lista `FERRAMENTAS` já convertida. Ferramentas MCP e ferramentas `@tool` locais podem ficar juntas na mesma lista.